# 12 · Tracing and Evaluation with LangSmith

Two problems every LLM app hits in production:

1. **Debugging blind** — a bad answer, no stack trace for "subtly wrong". → **Tracing**
2. **Invisible regressions** — a prompt tweak fixes 3 cases, silently breaks 5. → **Evaluation**

```
trace everything ─► bad runs surface ─► add to dataset ─► evaluate the fix ─► ship
       ▲                                                                        │
       └────────────────────────────────────────────────────────────────────────┘
```

> This notebook **runs without a LangSmith key** — the local evaluation works regardless; tracing lights up when you add the key.

---

## Setup

Tracing is configured entirely via environment — add to `../.env` (key from [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys):

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langchain-workspace
```

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv("../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)

configured = bool(os.getenv("LANGSMITH_API_KEY")) and os.getenv("LANGSMITH_TRACING") == "true"
print("LangSmith tracing:", "ON — runs will appear at smith.langchain.com" if configured
      else "off — local parts of this notebook still work; add the env vars above to trace")

---
## 1. Tracing: zero code changes

With the env vars set, **this ordinary chain call is already traced** — inputs, outputs, latency, token counts, per node. Nothing about the code says "tracing".

In [ ]:
qa = (
    ChatPromptTemplate.from_template("Answer as concisely as possible: {question}")
    | llm
    | StrOutputParser()
)

print(qa.invoke({"question": "Why do LLM apps need tracing in production?"}))

## 2. Pull plain Python into the trace: `@traceable`

Non-LangChain code (parsing, retrieval glue, business logic) joins the same trace tree with a decorator. Without the env vars it's a harmless no-op.

In [ ]:
from langsmith import traceable

@traceable
def normalize_question(raw: str) -> str:
    return raw.strip().rstrip("?").lower() + "?"

@traceable
def qa_pipeline(raw: str) -> str:
    return qa.invoke({"question": normalize_question(raw)})

print(qa_pipeline("  WHY is the sky blue "))

> In the LangSmith UI this shows as one tree: `qa_pipeline` → `normalize_question` + the chain (prompt → llm → parser), each with inputs/outputs and timing.

---
## 3. Evaluation, locally: dataset → target → judge

*Vibes are not a test suite.* The eval loop is a test suite where "passed" is a judgment call — so a model makes it (lesson 05's evaluator, promoted to offline QA). No platform needed for the pattern itself:

In [ ]:
dataset = [
    {"question": "What is the capital of Japan?",             "reference": "Tokyo"},
    {"question": "Which planet is closest to the sun?",       "reference": "Mercury"},
    {"question": "How many days are in a leap year?",         "reference": "366"},
    {"question": "Who wrote Romeo and Juliet?",               "reference": "William Shakespeare"},
    {"question": "What is the chemical symbol for gold?",     "reference": "Au"},
]

In [ ]:
class Grade(BaseModel):
    """Verdict on a single answer."""
    correct: bool = Field(description="Is the answer factually consistent with the reference?")
    reasoning: str = Field(description="One sentence explaining the verdict")

judge = (
    ChatPromptTemplate.from_template(
        "Question: {question}\n"
        "Reference answer: {reference}\n"
        "Submitted answer: {answer}\n\n"
        "Is the submitted answer factually consistent with the reference? "
        "Wording may differ; facts may not."
    )
    | llm.with_structured_output(Grade)
)

In [ ]:
passed = 0
for ex in dataset:
    answer = qa.invoke({"question": ex["question"]})
    grade = judge.invoke({**ex, "answer": answer})
    passed += grade.correct
    print(f"{'✅' if grade.correct else '❌'} {ex['question']}")
    print(f"     got: {answer!r}   expected: {ex['reference']!r}")

print(f"\nscore: {passed}/{len(dataset)}")

Run this before and after every prompt change and regressions stop being invisible. **Grow the dataset from real traces** — every production bug becomes a test case.

---
## 4. The hosted version: `evaluate()`

Same loop on LangSmith: results persisted, linked to traces, diffable across experiments. (Skipped automatically if no key is configured.)

In [ ]:
if not configured:
    print("Set LANGSMITH_API_KEY / LANGSMITH_TRACING in ../.env to run the hosted experiment.")
else:
    from langsmith import Client, evaluate

    client = Client()
    dataset_name = "workspace-qa-smoke"
    if not client.has_dataset(dataset_name=dataset_name):
        ds = client.create_dataset(dataset_name=dataset_name)
        client.create_examples(
            inputs=[{"question": ex["question"]} for ex in dataset],
            outputs=[{"reference": ex["reference"]} for ex in dataset],
            dataset_id=ds.id,
        )

    def target(inputs: dict) -> dict:
        return {"answer": qa.invoke({"question": inputs["question"]})}

    def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
        return judge.invoke({
            "question": inputs["question"],
            "answer": outputs["answer"],
            "reference": reference_outputs["reference"],
        }).correct

    results = evaluate(target, data=dataset_name, evaluators=[correctness],
                       experiment_prefix="lesson-12")

---
## What to remember

| Concept | What it does |
|---|---|
| 3 env vars | Every LangChain call traced — zero code changes |
| Trace | Tree of runs: inputs, outputs, latency, tokens, errors per node |
| `@traceable` | Plain Python functions join the trace tree |
| dataset → target → judge | The eval loop; ~30 lines with no platform at all |
| LLM-as-judge + structured output | A parseable verdict on quality |
| `evaluate(target, data, evaluators)` | Hosted experiments, diffable across versions |
| trace → dataset → evaluate → ship | The production feedback loop |

---
🏁 **Series complete.** Twelve patterns, one arc:

`invoke` → chains → parallel → routing → orchestration → evaluator loops → agents → thread & long-term memory → conversation views → RAG → multi-agent → streaming/async → observability.

Every one of them is a Runnable — which means everything composes with everything. That's the real lesson.